# Summary_Day14_online.ipynb  
## 사전학습 모델 활용 2 · 전이학습 전략 비교 · 인터넷 가능 버전

이번 14강은 **사전학습 모델을 어떻게 전략적으로 fine-tuning할 것인가**를 다루는 강의다.

13강에서는 사전학습 모델을 불러오고 마지막 분류기를 바꾸는 기본 흐름을 배웠다.  
14강에서는 한 단계 더 들어가서 다음 질문을 다룬다.

```text
어디까지 freeze할 것인가?
어느 layer부터 풀 것인가?
backbone과 head의 learning rate를 같게 줄 것인가 다르게 줄 것인가?
data augmentation은 어느 정도 강하게 줄 것인가?
작은 데이터셋에서는 어떤 전략이 안전한가?
```

강의 핵심 흐름은 다음이다.

```text
층별 동결 전략 비교
→ Full Freeze
→ Partial Fine-tuning
→ Full Fine-tuning
→ Overfit Gap 계산
→ 차등 학습률 Parameter Groups
→ Gradual Unfreezing
→ Weak / Medium / Strong Augmentation
→ 작은 데이터셋 전략 선택 가이드
```

이 파일은 **인터넷 가능 버전**이다.  
CIFAR-10 다운로드와 ImageNet 사전학습 ResNet18 가중치 다운로드가 가능하다는 전제로 작성했다.

> 필기 포인트:  
> 14강은 “모델을 어떻게 더 잘 학습시킬까”보다, “내 데이터 상황에서 어떤 전이학습 전략을 선택할까”에 더 가깝다.

## 1. 전체 실습 목적

이번 실습의 목적은 다음이다.

1. CIFAR-10을 작은 데이터셋 상황으로 줄여 실험한다.
2. ResNet18 사전학습 모델을 기준으로 전이학습 전략을 비교한다.
3. Full Freeze, Partial Fine-tuning, Full Fine-tuning의 차이를 코드로 확인한다.
4. `requires_grad`가 어떤 layer를 학습할지 결정하는 핵심임을 이해한다.
5. 단일 학습률과 차등 학습률을 비교한다.
6. `optimizer`의 parameter group 사용법을 익힌다.
7. weak, medium, strong augmentation의 차이를 정리한다.
8. 데이터 크기와 도메인 유사도에 따라 전략을 고르는 기준을 정리한다.

## 2. 라이브러리 준비

### 함수/모듈 사용법

```python
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
```

- `torch`: Tensor 계산과 GPU 사용에 필요하다.
- `nn`: `Linear`, `CrossEntropyLoss` 같은 신경망 구성 요소를 만든다.
- `optim`: Adam, SGD 같은 Optimizer를 만든다.
- `datasets`: CIFAR-10 데이터를 다운로드한다.
- `transforms`: Resize, Crop, Flip, Normalize 등 이미지 전처리를 만든다.
- `models`: ResNet18 같은 사전학습 모델을 불러온다.
- `DataLoader`: mini-batch 단위로 데이터를 꺼낸다.
- `Subset`: 전체 데이터 중 일부 index만 뽑아 작은 데이터셋을 만든다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

print("PyTorch:", torch.__version__)

## 3. device 설정과 class 이름

### 함수 사용법

```python
torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

- GPU가 있으면 `cuda`를 사용한다.
- GPU가 없으면 `cpu`를 사용한다.
- 모델과 Tensor는 같은 device에 있어야 한다.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

num_classes = len(class_names)

print("사용 device:", device)
print("클래스 수:", num_classes)

## 4. 전이학습 전략 3가지

14강에서 가장 중요한 비교 대상은 3가지다.

| 전략 | 설명 | 장점 | 위험 |
|---|---|---|---|
| Full Freeze | backbone 전체 동결, classifier만 학습 | 빠르고 과적합 위험 작음 | 새 도메인 적응력이 낮을 수 있음 |
| Partial Fine-tuning | 마지막 block 일부와 classifier 학습 | 성능과 안정성의 균형 | 어느 block을 풀지 선택 필요 |
| Full Fine-tuning | 전체 layer 학습 | 데이터가 많으면 가장 유연함 | 작은 데이터에서는 과적합 위험 큼 |

핵심 코드는 전부 이것에서 시작한다.

```python
param.requires_grad = False
param.requires_grad = True
```

> 강의 포인트:  
> 전이학습에는 정답이 하나로 고정되어 있지 않다.  
> 데이터 크기, 도메인 유사도, 과적합 정도를 보고 전략을 선택해야 한다.

## 5. 기본 transform 만들기

사전학습 ResNet은 ImageNet 기준으로 학습되었기 때문에 입력 크기와 정규화를 맞춰주는 것이 좋다.

### 함수 사용법

```python
transforms.Resize(224)
transforms.ToTensor()
transforms.Normalize(mean, std)
```

- `Resize(224)`: 이미지를 224×224로 맞춘다.
- `ToTensor()`: PIL 이미지를 Tensor로 바꾼다.
- `Normalize()`: ImageNet 평균과 표준편차 기준으로 정규화한다.

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

transform_basic = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

print(transform_basic)

## 6. CIFAR-10 다운로드

### 함수 사용법

```python
datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_basic)
```

- `root`: 데이터 저장 위치다.
- `train=True`: 학습 데이터다.
- `train=False`: 테스트 데이터다.
- `download=True`: 데이터가 없으면 다운로드한다.
- `transform`: 이미지에 적용할 전처리다.

In [ ]:
full_train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_basic
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_basic
)

print("전체 학습 데이터:", len(full_train_dataset))
print("테스트 데이터:", len(test_dataset))

## 7. 작은 데이터셋 만들기

강의에서는 작은 데이터셋 상황을 만들기 위해 class당 일부 샘플만 뽑는다.

### 코드 흐름

```text
class_indices = {0: [], 1: [], ..., 9: []}
전체 train dataset 순회
각 class마다 samples_per_class개까지만 index 저장
selected_indices에 전부 합치기
Subset으로 작은 dataset 생성
```

작은 데이터셋을 일부러 만드는 이유는 transfer learning 전략 차이가 더 잘 드러나기 때문이다.

In [ ]:
samples_per_class = 50

class_indices = {i: [] for i in range(num_classes)}

for idx, (_, label) in enumerate(full_train_dataset):
    if len(class_indices[label]) < samples_per_class:
        class_indices[label].append(idx)

selected_indices = []

for indices in class_indices.values():
    selected_indices.extend(indices)

small_train_dataset = Subset(full_train_dataset, selected_indices)

print("small dataset size:", len(small_train_dataset))
print("sample size per class:", samples_per_class)

for class_id, indices in class_indices.items():
    print(f"{class_names[class_id]:10s}: {len(indices)}개")

## 8. DataLoader 만들기

### 함수 사용법

```python
DataLoader(dataset, batch_size=32, shuffle=True)
```

- `batch_size`: 한 번에 학습할 샘플 수다.
- `shuffle=True`: 학습 데이터 순서를 섞는다.
- test loader는 보통 `shuffle=False`로 둔다.

In [ ]:
batch_size = 32

train_loader = DataLoader(
    small_train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print("학습 batch 수:", len(train_loader))
print("테스트 batch 수:", len(test_loader))

## 9. 샘플 이미지 시각화

정규화된 이미지는 바로 보면 색이 이상할 수 있다.  
그래서 `img = std * img + mean`으로 역정규화한다.

In [ ]:
def denormalize(img):
    img = img.numpy().transpose((1, 2, 0))
    mean = np.array(imagenet_mean)
    std = np.array(imagenet_std)
    img = std * img + mean
    img = np.clip(img, 0, 1)
    return img


def show_sample_images(dataset, num_images=5):
    fig, axes = plt.subplots(1, num_images, figsize=(15, 3))

    for i in range(num_images):
        img, label = dataset[i]

        axes[i].imshow(denormalize(img))
        axes[i].set_title(class_names[label])
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()

show_sample_images(small_train_dataset, num_images=5)

## 10. ResNet18 불러오기 함수

온라인 버전은 ImageNet pretrained weights를 사용한다.

### 함수 사용법

최신 torchvision:

```python
models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
```

구버전 호환:

```python
models.resnet18(pretrained=True)
```

둘 다 ImageNet으로 학습된 가중치를 불러오는 목적이다.

In [ ]:
def load_resnet18_pretrained():
    try:
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    except Exception:
        model = models.resnet18(pretrained=True)

    return model

tmp_model = load_resnet18_pretrained()

print(tmp_model.fc)
print("fc input features:", tmp_model.fc.in_features)

## 11. build_model 함수 만들기

`build_model(strategy)`는 전략 이름에 따라 ResNet18의 학습 가능한 layer를 다르게 설정한다.

### 함수 사용법

```python
model = build_model("freeze")
model = build_model("partial")
model = build_model("full")
```

- `"freeze"`: 전체 backbone을 동결하고 `fc`만 학습한다.
- `"partial"`: 전체를 동결한 뒤 `layer4`와 `fc`만 학습한다.
- `"full"`: 전체 layer를 학습한다.

> 주의:  
> Full Fine-tuning은 optimizer에도 전체 `model.parameters()`를 넣어야 의미가 맞다.

In [ ]:
def count_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def build_model(strategy):
    model = load_resnet18_pretrained()

    if strategy == "freeze":
        for param in model.parameters():
            param.requires_grad = False

        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif strategy == "partial":
        for param in model.parameters():
            param.requires_grad = False

        for param in model.layer4.parameters():
            param.requires_grad = True

        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif strategy == "full":
        for param in model.parameters():
            param.requires_grad = True

        model.fc = nn.Linear(model.fc.in_features, num_classes)

    else:
        raise ValueError("strategy는 freeze, partial, full 중 하나여야 한다.")

    return model.to(device)


for strategy in ["freeze", "partial", "full"]:
    model = build_model(strategy)
    total, trainable = count_trainable_params(model)
    print(f"{strategy:8s} | total={total:,} | trainable={trainable:,}")

## 12. 학습 함수와 평가 함수 만들기

### 함수 사용법

```python
train_one_epoch(model, dataloader, criterion, optimizer, device)
evaluate_model(model, dataloader, criterion, device)
```

- `train_one_epoch`: 한 epoch 동안 학습한다.
- `evaluate_model`: 평가 데이터에서 loss와 accuracy를 계산한다.
- 학습 함수에서는 `model.train()`을 사용한다.
- 평가 함수에서는 `model.eval()`과 `torch.no_grad()`를 사용한다.

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

        _, predicted = outputs.max(1)

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc


def evaluate_model(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)

            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    eval_loss = running_loss / total
    eval_acc = 100.0 * correct / total

    return eval_loss, eval_acc

## 13. 전략별 Optimizer 만들기

전략마다 학습 대상 parameter가 다르다.

```text
freeze → model.fc.parameters()
partial → layer4 + fc
full → model.parameters()
```

### 함수 사용법

```python
optim.Adam([...], lr=learning_rate)
```

- 리스트 안에 여러 parameter group을 넣을 수 있다.
- group마다 다른 learning rate를 줄 수도 있다.

In [ ]:
learning_rate = 0.001

def make_optimizer(model, strategy, learning_rate=0.001):
    if strategy == "freeze":
        optimizer = optim.Adam(model.fc.parameters(), lr=learning_rate)

    elif strategy == "partial":
        optimizer = optim.Adam(
            [
                {"params": model.layer4.parameters()},
                {"params": model.fc.parameters()}
            ],
            lr=learning_rate
        )

    elif strategy == "full":
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    else:
        raise ValueError("strategy는 freeze, partial, full 중 하나여야 한다.")

    return optimizer

## 14. 전략 하나를 학습하는 함수

반복되는 코드를 줄이기 위해 `run_strategy()` 함수를 만든다.

### 함수 사용법

```python
history, elapsed_time = run_strategy("freeze", num_epochs=2)
```

- `history`: train loss, train accuracy, test accuracy 기록이다.
- `elapsed_time`: 학습에 걸린 시간이다.

In [ ]:
def run_strategy(strategy, num_epochs=2):
    model = build_model(strategy)
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(model, strategy, learning_rate)

    history = {
        "train_loss": [],
        "train_acc": [],
        "test_acc": []
    }

    start_time = time.time()

    print(f"\n[{strategy}] 학습 시작")

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        _, test_acc = evaluate_model(
            model,
            test_loader,
            criterion,
            device
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)

        print(
            f"Epoch [{epoch + 1}/{num_epochs}] - "
            f"Train Loss: {train_loss:.4f}, "
            f"Train Acc: {train_acc:.2f}%, "
            f"Test Acc: {test_acc:.2f}%"
        )

    elapsed_time = time.time() - start_time

    print(f"학습 소요 시간: {elapsed_time:.2f}초")

    return history, elapsed_time

## 15. 세 가지 동결 전략 실행

실제 강의 흐름은 세 전략을 모두 실행해 비교한다.

실습 시간이 길면 `num_epochs`를 줄이면 된다.  
아래 기본값은 빠른 실행을 위해 1 epoch로 둔다.

In [ ]:
num_epochs = 1

history_freeze, elapsed_time_freeze = run_strategy("freeze", num_epochs=num_epochs)
history_partial, elapsed_time_partial = run_strategy("partial", num_epochs=num_epochs)
history_full, elapsed_time_full = run_strategy("full", num_epochs=num_epochs)

## 16. 동결 전략 결과 비교 그래프

비교할 지표는 다음이다.

```text
train accuracy
test accuracy
overfit gap = train accuracy - test accuracy
training time
```

과적합 gap이 크면 train 데이터에는 잘 맞지만 test 일반화가 약할 수 있다.

In [ ]:
epochs = range(1, num_epochs + 1)

plt.plot(epochs, history_freeze["train_acc"], "b-o", label="Full Freeze Train")
plt.plot(epochs, history_partial["train_acc"], "g-s", label="Partial Train")
plt.plot(epochs, history_full["train_acc"], "r-^", label="Full Train")
plt.xlabel("Epoch")
plt.ylabel("Train Accuracy (%)")
plt.title("Training Accuracy Comparison")
plt.legend()
plt.show()

plt.plot(epochs, history_freeze["test_acc"], "b-o", label="Full Freeze Test")
plt.plot(epochs, history_partial["test_acc"], "g-s", label="Partial Test")
plt.plot(epochs, history_full["test_acc"], "r-^", label="Full Test")
plt.xlabel("Epoch")
plt.ylabel("Test Accuracy (%)")
plt.title("Test Accuracy Comparison")
plt.legend()
plt.show()

In [ ]:
results = [
    [
        "Full Freeze",
        history_freeze["test_acc"][-1],
        elapsed_time_freeze,
        history_freeze["train_acc"][-1] - history_freeze["test_acc"][-1]
    ],
    [
        "Partial Fine-tune",
        history_partial["test_acc"][-1],
        elapsed_time_partial,
        history_partial["train_acc"][-1] - history_partial["test_acc"][-1]
    ],
    [
        "Full Fine-tune",
        history_full["test_acc"][-1],
        elapsed_time_full,
        history_full["train_acc"][-1] - history_full["test_acc"][-1]
    ]
]

print(f'{"Strategy":<20} {"Test Acc (%)":<15} {"Time (sec)":<15} {"Overfit Gap (%)":<15}')
print("-" * 80)

for result in results:
    print(f"{result[0]:<20} {result[1]:<15.2f} {result[2]:<15.2f} {result[3]:<15.2f}")

## 17. 차등 학습률 Differential Learning Rate

차등 학습률은 backbone과 head에 서로 다른 learning rate를 주는 방법이다.

전이학습에서는 보통 다음 감각으로 설정한다.

```text
backbone: 작게 학습
head/classifier: 크게 학습
```

이유는 다음이다.

- backbone은 이미 ImageNet에서 좋은 특징을 배웠다.
- 너무 큰 learning rate를 주면 기존 지식이 망가질 수 있다.
- classifier는 새로 만든 layer라 더 빨리 학습해야 한다.

### 함수 사용법

```python
optimizer = optim.Adam([
    {"params": model.layer4.parameters(), "lr": 0.001},
    {"params": model.fc.parameters(), "lr": 0.01}
])
```

parameter group마다 `lr`을 따로 지정한다.

In [ ]:
def build_partial_model():
    model = load_resnet18_pretrained()

    for param in model.parameters():
        param.requires_grad = False

    for param in model.layer4.parameters():
        param.requires_grad = True

    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model.to(device)


model_uniform_lr = build_partial_model()

optimizer_uniform = optim.Adam(
    [
        {"params": model_uniform_lr.layer4.parameters()},
        {"params": model_uniform_lr.fc.parameters()}
    ],
    lr=0.001
)

model_diff_lr = build_partial_model()

optimizer_diff = optim.Adam(
    [
        {"params": model_diff_lr.layer4.parameters(), "lr": 0.001},
        {"params": model_diff_lr.fc.parameters(), "lr": 0.01}
    ]
)

print("단일 학습률 group:")
for group in optimizer_uniform.param_groups:
    print(group["lr"])

print("\n차등 학습률 group:")
for group in optimizer_diff.param_groups:
    print(group["lr"])

## 18. Gradual Unfreezing 개념

Gradual Unfreezing은 처음에는 head만 학습하다가, 점점 뒤쪽 layer부터 풀어가는 전략이다.

예시는 다음이다.

```text
epoch 1~5: head만 학습
epoch 6~10: head + layer4
epoch 11~15: head + layer4 + layer3
epoch 16 이후: 더 앞 layer까지 점진적으로 해제
```

작은 데이터셋에서 한 번에 모든 layer를 풀면 과적합이 심해질 수 있다.  
그래서 점진적으로 푸는 방식이 안전할 수 있다.

In [ ]:
def freeze_all(model):
    for param in model.parameters():
        param.requires_grad = False


def unfreeze_module(module):
    for param in module.parameters():
        param.requires_grad = True


def build_gradual_model():
    model = load_resnet18_pretrained()
    freeze_all(model)

    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model.to(device)


def apply_gradual_unfreeze(model, stage):
    if stage == "head":
        freeze_all(model)
        unfreeze_module(model.fc)

    elif stage == "layer4":
        freeze_all(model)
        unfreeze_module(model.layer4)
        unfreeze_module(model.fc)

    elif stage == "layer3_layer4":
        freeze_all(model)
        unfreeze_module(model.layer3)
        unfreeze_module(model.layer4)
        unfreeze_module(model.fc)

    elif stage == "all":
        for param in model.parameters():
            param.requires_grad = True

    else:
        raise ValueError("stage가 올바르지 않다.")

    total, trainable = count_trainable_params(model)
    print(f"stage={stage:13s} | trainable={trainable:,} / {total:,}")


gradual_model = build_gradual_model()

for stage in ["head", "layer4", "layer3_layer4", "all"]:
    apply_gradual_unfreeze(gradual_model, stage)

## 19. 데이터 증강 강도 3가지

강의에서는 증강 강도를 weak, medium, strong으로 나눠 비교했다.

| 강도 | 대표 변환 | 특징 |
|---|---|---|
| Weak | Resize, HorizontalFlip | 기본적이고 안전함 |
| Medium | RandomResizedCrop, ColorJitter | 일반화 성능을 높일 수 있음 |
| Strong | Rotation, RandomErasing, AutoAugment 계열 | 데이터가 많을 때 효과적일 수 있음 |

> 강의 포인트:  
> 증강은 무조건 강할수록 좋은 것이 아니다.  
> 데이터가 너무 적거나 의료 영상처럼 디테일이 중요한 도메인에서는 강한 증강이 오히려 성능을 떨어뜨릴 수 있다.

In [ ]:
transform_weak = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

transform_medium = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

transform_strong = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
    transforms.RandomErasing(p=0.5)
])

print("Weak Augmentation:")
print(transform_weak)

print("\nMedium Augmentation:")
print(transform_medium)

print("\nStrong Augmentation:")
print(transform_strong)

## 20. 증강 강도별 DataLoader 만들기

같은 `selected_indices`를 사용하되 transform만 다르게 적용한다.  
이렇게 해야 데이터 구성은 같고 증강 강도만 비교할 수 있다.

In [ ]:
train_dataset_weak = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_weak
)

train_dataset_medium = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_medium
)

train_dataset_strong = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_strong
)

small_train_weak = Subset(train_dataset_weak, selected_indices)
small_train_medium = Subset(train_dataset_medium, selected_indices)
small_train_strong = Subset(train_dataset_strong, selected_indices)

loader_weak = DataLoader(small_train_weak, batch_size=batch_size, shuffle=True, num_workers=0)
loader_medium = DataLoader(small_train_medium, batch_size=batch_size, shuffle=True, num_workers=0)
loader_strong = DataLoader(small_train_strong, batch_size=batch_size, shuffle=True, num_workers=0)

print("weak batch:", len(loader_weak))
print("medium batch:", len(loader_medium))
print("strong batch:", len(loader_strong))

## 21. 증강 강도 선택 가이드

강의 기준으로 정리하면 다음과 같다.

```text
작은 데이터셋:
    weak 또는 medium부터 시작한다.

중간 데이터셋:
    medium을 기본으로 두고 strong도 비교한다.

큰 데이터셋:
    strong augmentation이 효과를 낼 수 있다.

의료 영상/정밀 도메인:
    강한 왜곡은 조심한다.
```

테스트 데이터와 검증 데이터에는 증강을 적용하지 않는다.  
검증은 실제 평가 기준이어야 하므로, 학습용으로만 랜덤 변형을 준다.

## 22. 작은 데이터셋 전략 선택 가이드

PDF와 강의 스크립트의 의사결정 흐름을 정리하면 다음이다.

| Target Data Size | 권장 전략 |
|---|---|
| 매우 작음, 전체 1000개 미만 | Full Freeze부터 시작 |
| 중간, 1K~10K | Partial Fine-tuning, 도메인 유사하면 마지막 1~2 block |
| 큼, 10K 이상 | Full Fine-tuning 가능, 강한 regularization 필요 |

추가로 봐야 할 기준:

- 도메인이 ImageNet과 비슷한가?
- train/test gap이 큰가?
- validation curve가 안정적인가?
- augmentation이 원본 의미를 깨지 않는가?
- class imbalance가 심하지 않은가?

> 실무식 결론:  
> 작은 데이터셋에서는 빠르게 full fine-tuning으로 가지 말고, freeze → partial → full 순서로 실험하는 것이 안전하다.

## 23. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `freeze` | 파라미터 동결 | `requires_grad=False` |
| `unfreeze` | 파라미터 학습 허용 | `requires_grad=True` |
| `Full Freeze` | backbone 전체 동결 | fc만 학습 |
| `Partial Fine-tuning` | 일부 block만 해제 | layer4 + fc 학습 |
| `Full Fine-tuning` | 전체 layer 학습 | 과적합 주의 |
| `backbone` | 특징 추출기 | ResNet의 conv/layer 부분 |
| `head` | 마지막 분류기 | ResNet의 fc |
| `layer4` | ResNet 마지막 block | partial 전략에서 자주 해제 |
| `fc` | fully connected layer | ResNet의 마지막 분류기 |
| `requires_grad` | gradient 계산 여부 | 학습 여부 제어 |
| `param_groups` | optimizer parameter group | group별 lr 지정 |
| `differential LR` | 차등 학습률 | backbone 작게, head 크게 |
| `overfit gap` | train acc - test acc | 과적합 정도 |
| `augmentation` | 데이터 증강 | 학습 데이터 변형 |
| `RandomResizedCrop` | 랜덤 crop 후 resize | medium/strong 증강 |
| `ColorJitter` | 밝기/대비/색상 변형 | 이미지 색 변화 |
| `RandomErasing` | 일부 영역 지우기 | strong 증강 |
| `Gradual Unfreezing` | 점진적 layer 해제 | head부터 차례로 해제 |

## 24. 시험용 요약

```text
14강 핵심 = 전이학습에서 어떤 layer를 학습할지, 어떤 learning rate와 augmentation을 쓸지 전략적으로 선택하는 것
```

꼭 기억할 것:

- `requires_grad=False`는 해당 parameter를 학습하지 않겠다는 뜻이다.
- `requires_grad=True`는 해당 parameter를 학습하겠다는 뜻이다.
- Full Freeze는 backbone을 모두 동결하고 classifier만 학습한다.
- Partial Fine-tuning은 마지막 block 일부와 classifier를 학습한다.
- Full Fine-tuning은 전체 layer를 학습한다.
- 작은 데이터셋에서는 Full Freeze 또는 Partial부터 시작하는 것이 안전하다.
- 큰 데이터셋에서는 Full Fine-tuning도 고려할 수 있다.
- Overfit Gap은 `Train Acc - Test Acc`다.
- Gap이 크면 과적합을 의심한다.
- 차등 학습률은 backbone에는 작은 lr, head에는 큰 lr을 주는 전략이다.
- optimizer parameter group을 사용하면 layer별 lr을 다르게 줄 수 있다.
- Gradual Unfreezing은 head부터 시작해 점진적으로 layer를 해제하는 방법이다.
- Weak Augmentation은 기본적이고 안전하다.
- Medium Augmentation은 작은/중간 데이터셋에서 많이 쓰기 좋다.
- Strong Augmentation은 데이터가 많을 때 효과적일 수 있지만 왜곡 위험이 있다.
- validation/test 데이터에는 random augmentation을 적용하지 않는다.